# Redes Neurais (MLP) — Otimização de Hiperparâmetros, Ablação e Equidade

Este notebook consolida e aprofunda os experimentos com **Redes Neurais Multicamadas (*Multi-Layer Perceptron — MLP*)** para predição de doença cardíaca na base *Heart Failure Prediction* (918 pacientes).

### Estrutura do Estudo:
1. **Tratamento de Dados e Pipeline Clínico:** Auditoria dos ausentes ocultos (`colesterol == 0` e `pressao_repouso == 0`).
2. **Busca Sistemática de Hiperparâmetros:** Otimização via `GridSearchCV` (topologias, regularização L2 e taxa de aprendizado).
3. **Avaliação no Teste Reservado:** Métricas de classificação, matriz de confusão e curva ROC.
4. **Experimentos de Ablação Clínica:** Impacto da remoção de sintomas-chave (`ChestPainType`, `ExerciseAngina`, `Oldpeak`, `ST_Slope`).
5. **Avaliação de Equidade e Subgrupos:** Análise por Sexo (Mulheres vs. Homens) e Faixas Etárias (<45, 45–59, 60+).
6. **Interpretabilidade e Espaço Latente:** *Permutation Feature Importance* e projeção PCA das ativações da camada oculta.
7. **Síntese e Comparação:** Síntese frente aos modelos de Árvore de Decisão e Random Forest.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
SEED = 42


## 1. Carga dos Dados e Tratamento dos Ausentes Ocultos

A análise exploratória revelou que os valores iguais a zero em `Cholesterol` (172 casos) e `RestingBP` (1 caso) são fisiologicamente impossíveis e decorrem de protocolos hospitalares específicos. 

Para evitar vazamento de dados (*data leakage*), convertemos esses zeros em `NaN` e os imputamos com a mediana estritamente dentro de cada dobra de treinamento do pipeline.


In [ ]:
# Localização da base de dados com fallback
caminhos = [Path("data/heart.csv"), Path("heart.csv"), Path("../data/heart.csv")]
caminho_csv = next((p for p in caminhos if p.exists()), None)

if caminho_csv:
    df = pd.read_csv(caminho_csv)
else:
    url = "https://raw.githubusercontent.com/jayachandru001/Heart-Failure-Prediction-/main/heart.csv"
    df = pd.read_csv(url)

print(f"Dimensões originais: {df.shape} (linhas × colunas)")

# Conversão de zeros espúrios para NaN
zeros_chol = (df['Cholesterol'] == 0).sum()
zeros_bp = (df['RestingBP'] == 0).sum()
df.loc[df['Cholesterol'] == 0, 'Cholesterol'] = np.nan
df.loc[df['RestingBP'] == 0, 'RestingBP'] = np.nan

print(f"Colesterol: {zeros_chol} zeros convertidos em NaN ({zeros_chol/len(df):.1%})")
print(f"Pressão de repouso: {zeros_bp} zero convertido em NaN")
print(f"Distribuição do alvo: {df['HeartDisease'].value_counts(normalize=True).to_dict()}")
df.head()


## 2. Preparação do Pipeline e Busca de Hiperparâmetros

Dividimos os dados em **80% treino (734)** e **20% teste reservado (184)**, estratificados pela variável-alvo `HeartDisease`.

Otimizamos a topologia e a regularização da rede neural utilizando `GridSearchCV` com validação cruzada 5-fold estratificada.


In [ ]:
X = df.drop(columns='HeartDisease')
y = df['HeartDisease']

cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
num_cols = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']

# Divisão treino/teste reservado (mesma partição dos outros notebooks)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

pipe = Pipeline([
    ('prep', preprocessor),
    ('mlp', MLPClassifier(
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=SEED
    ))
])

# Grade de hiperparâmetros
param_grid = {
    'mlp__hidden_layer_sizes': [(32,), (16, 8), (32, 16), (64, 32)],
    'mlp__alpha': [0.001, 0.01, 0.1],
    'mlp__learning_rate_init': [0.001, 0.01]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print(f"\nMelhores hiperparâmetros encontrados:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")
print(f"F1-Score médio na validação cruzada: {grid.best_score_:.4f}")

# Top 5 combinações
res_grid = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score').head(5)
res_grid[['params', 'mean_test_score', 'std_test_score']]


## 3. Avaliação no Teste Reservado

Avaliamos o modelo final com os melhores hiperparâmetros no conjunto de teste independente (184 pacientes).


In [ ]:
best_mlp = grid.best_estimator_

y_pred = best_mlp.predict(X_test)
y_prob = best_mlp.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("="*50)
print("DESEMPENHO NO TESTE RESERVADO")
print("="*50)
print(f"Acurácia      : {acc:.4f}")
print(f"Precisão      : {prec:.4f}")
print(f"Sensibilidade : {rec:.4f} (Recall da classe positiva)")
print(f"F1-Score      : {f1:.4f}")
print(f"ROC-AUC       : {auc:.4f}")
print("\nRelatório de Classificação:\n")
print(classification_report(y_test, y_pred, target_names=['Sem Doença (0)', 'Com Doença (1)']))

# Visualização: Matriz de Confusão e Curva ROC
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax1,
            xticklabels=['Previsto: 0', 'Previsto: 1'],
            yticklabels=['Real: 0', 'Real: 1'])
ax1.set_title(f'Matriz de Confusão (Falsos Negativos: {cm[1, 0]})')

fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color='#1f77b4', lw=2, label=f'MLP (AUC = {auc:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=1)
ax2.set_xlabel('Taxa de Falsos Positivos')
ax2.set_ylabel('Taxa de Verdadeiros Positivos')
ax2.set_title('Curva ROC')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()


## 4. Experimentos de Ablação de Atributos Clínicos

Avalia-se a resiliência da rede neural à ausência de informações sintomáticas e eletrocardiográficas chave:
* **M1**: Modelo Completo (todas as 11 variáveis).
* **M2**: Sem `ChestPainType` (dor torácica).
* **M3**: Sem `ExerciseAngina` (angina de esforço).
* **M4**: Sem `ChestPainType` e sem `ExerciseAngina`.
* **M5**: Sem `Oldpeak` (depressão de ST).
* **M6**: Sem `ST_Slope` (inclinação de ST).


In [ ]:
def avaliar_ablatacao(nome, remover=[]):
    cur_cat = [c for c in cat_cols if c not in remover]
    cur_num = [c for c in num_cols if c not in remover]
    
    prep = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), cur_num),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cur_cat)
    ])
    
    # Clona a melhor configuração de hiperparâmetros encontrada
    best_params = {k.replace('mlp__', ''): v for k, v in grid.best_params_.items()}
    mlp_model = MLPClassifier(max_iter=500, early_stopping=True, random_state=SEED, **best_params)
    
    pipeline_abl = Pipeline([('prep', prep), ('mlp', mlp_model)])
    pipeline_abl.fit(X_train.drop(columns=remover), y_train)
    
    preds = pipeline_abl.predict(X_test.drop(columns=remover))
    probs = pipeline_abl.predict_proba(X_test.drop(columns=remover))[:, 1]
    
    return {
        'Cenário': nome,
        'N_Var': len(cur_cat) + len(cur_num),
        'Acurácia': accuracy_score(y_test, preds),
        'Sensibilidade': recall_score(y_test, preds),
        'Precisão': precision_score(y_test, preds),
        'F1_Score': f1_score(y_test, preds),
        'ROC_AUC': roc_auc_score(y_test, probs)
    }

cenarios = [
    ("M1 - Completo", []),
    ("M2 - Sem ChestPainType", ["ChestPainType"]),
    ("M3 - Sem ExerciseAngina", ["ExerciseAngina"]),
    ("M4 - Sem Dor + Angina", ["ChestPainType", "ExerciseAngina"]),
    ("M5 - Sem Oldpeak", ["Oldpeak"]),
    ("M6 - Sem ST_Slope", ["ST_Slope"])
]

df_ablacao = pd.DataFrame([avaliar_ablatacao(nome, rem) for nome, rem in cenarios])
display(df_ablacao.round(4))

# Gráfico comparativo de impacto
fig, ax = plt.subplots(figsize=(10, 4.5))
df_abl_plot = df_ablacao.sort_values('F1_Score', ascending=True)
ax.barh(df_abl_plot['Cenário'], df_abl_plot['F1_Score'], color='#2ca02c', alpha=0.85, label='F1-Score')
ax.set_xlim(0.70, 0.95)
ax.set_xlabel('F1-Score no Teste Reservado')
ax.set_title('Impacto da Ablação de Atributos Clínicos na Rede Neural')
for i, v in enumerate(df_abl_plot['F1_Score']):
    ax.text(v + 0.005, i, f"{v:.4f}", va='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()


## 5. Avaliação de Equidade e Subgrupos Demográficos

Uma questão central para a segurança clínica: **a rede neural se comporta da mesma forma para diferentes públicos?**

Avaliamos a capacidade de ordenamento de risco (ROC-AUC), a sensibilidade e a especificidade por **sexo** e por **faixa etária** sob o limiar fixo clássico de 0,5.


In [ ]:
# Previsões no conjunto de teste completo
df_test = X_test.copy()
df_test['y_true'] = y_test.values
df_test['y_pred'] = y_pred
df_test['y_prob'] = y_prob
df_test['AgeBand'] = pd.cut(df_test['Age'], bins=[0, 44, 59, 120], labels=['<45', '45-59', '60+'])

grupos = {
    'Todos': np.ones(len(df_test), dtype=bool),
    'Mulheres': (df_test['Sex'] == 'F').values,
    'Homens': (df_test['Sex'] == 'M').values,
    'Idade <45': (df_test['AgeBand'] == '<45').values,
    'Idade 45-59': (df_test['AgeBand'] == '45-59').values,
    'Idade 60+': (df_test['AgeBand'] == '60+').values
}

linhas_sub = []
for nome, mask in grupos.items():
    sub = df_test[mask]
    n = len(sub)
    n_pos = (sub['y_true'] == 1).sum()
    prev = n_pos / n if n > 0 else np.nan
    prob_m = sub['y_prob'].mean()
    
    auc_g = roc_auc_score(sub['y_true'], sub['y_prob']) if len(np.unique(sub['y_true'])) == 2 else np.nan
    sens_g = recall_score(sub['y_true'], sub['y_pred'], zero_division=0)
    spec_g = recall_score(sub['y_true'], sub['y_pred'], pos_label=0, zero_division=0)
    
    linhas_sub.append({
        'Subgrupo': nome,
        'N': n,
        'Casos Doença': n_pos,
        'Prevalência Real': prev,
        'Prob. Prevista Média': prob_m,
        'ROC-AUC': auc_g,
        'Sensibilidade': sens_g,
        'Especificidade': spec_g
    })

tabela_subgrupos = pd.DataFrame(linhas_sub)
display(tabela_subgrupos.round(3))

# Gráfico de disparidade Sensibilidade vs Especificidade por público
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

sns.barplot(data=tabela_subgrupos[tabela_subgrupos['Subgrupo'].isin(['Mulheres', 'Homens'])], 
            x='Subgrupo', y='Sensibilidade', palette=['#c4571f', '#1f6fb5'], ax=ax1)
ax1.set_ylim(0.5, 1.0)
ax1.set_title('Sensibilidade por Sexo (Limiar 0,5)\nMulheres apresentam maior taxa de Falsos Negativos')

sns.barplot(data=tabela_subgrupos[tabela_subgrupos['Subgrupo'].str.startswith('Idade')], 
            x='Subgrupo', y='Sensibilidade', palette='Blues_r', ax=ax2)
ax2.set_ylim(0.5, 1.0)
ax2.set_title('Sensibilidade por Faixa Etária (Limiar 0,5)\nSubdiagnóstico em pacientes jovens (<45 anos)')

plt.tight_layout()
plt.show()


## 6. Interpretabilidade: O que a Rede Aprendeu?

1. **Permutation Feature Importance (PFI):** Mensura a perda de acurácia ao embaralhar cada atributo no teste.
2. **Espaço Latente da Camada Oculta (PCA):** Extração das representações intermediárias dos neurônios para visualizar como a rede separa os pacientes geometricamente.


In [ ]:
# 1. Permutation Importance
r_imp = permutation_importance(best_mlp, X_test, y_test, scoring='f1', n_repeats=20, random_state=SEED)
df_imp = pd.DataFrame({
    'Atributo': X_test.columns,
    'Importância_Média': r_imp.importances_mean,
    'Desvio': r_imp.importances_std
}).sort_values('Importância_Média', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_imp.head(8), x='Importância_Média', y='Atributo', color='#3b528b', ax=ax1)
ax1.set_title('Top 8 Preditores Mais Determinantes (Queda de F1)')
ax1.set_xlabel('Queda Média no F1-Score')

# 2. Espaço Latente (Representação Intermediária da Camada Oculta)
mlp_obj = best_mlp.named_steps['mlp']
prep_obj = best_mlp.named_steps['prep']

X_proc = prep_obj.transform(X_test)
# Computa ativações da primeira camada oculta: relu(X * W0 + b0)
W0, b0 = mlp_obj.coefs_[0], mlp_obj.intercepts_[0]
ativacoes = np.maximum(0, np.dot(X_proc, W0) + b0)

pca = PCA(n_components=2, random_state=SEED)
latent_2d = pca.fit_transform(ativacoes)

scatter = ax2.scatter(latent_2d[:, 0], latent_2d[:, 1], c=y_test, cmap='coolwarm', alpha=0.85, edgecolors='k', s=50)
ax2.set_title(f'Espaço Latente da Rede (PCA das Ativações Ocultas)\nVariância Explicada: {pca.explained_variance_ratio_.sum():.1%}')
ax2.set_xlabel('Componente Principal 1')
ax2.set_ylabel('Componente Principal 2')
plt.colorbar(scatter, ax=ax2, label='Diagnóstico Real (0: Saudável, 1: Doente)')

plt.tight_layout()
plt.show()


## 7. Síntese dos Resultados e Conclusões Clínicas

### 1. Desempenho e Hiperparâmetros
* A arquitetura ótima identificada pela busca em grade foi **`(32, 16)` neurônios com taxa inicial de $0{,}01$ e regularização L2 $\alpha = 0{,}001$**, atingindo **F1 de $0{,}8812$ e ROC-AUC de $0{,}9322$** no teste independente.
* Posiciona-se competitivamente entre a **Árvore de Decisão** ($F_1 = 0{,}8387$, Acurácia $= 80{,}98\%$) e o **Random Forest** ($F_1 = 0{,}9091$, Acurácia $= 89{,}67\%$).

### 2. Ablação de Atributos
* A exclusão de `ST_Slope` e `ChestPainType` produz as maiores quedas no F1-Score e na Sensibilidade da rede, evidenciando que os dados eletrocardiográficos e sintomáticos de esforço formam o núcleo de decisão do modelo.

### 3. Equidade e Calibração de Limiar
* A capacidade de ordenamento de risco da rede neural é excelente e similar em ambos os sexos ($AUC \approx 0{,}91$).
* Contudo, sob o ponto de corte uniforme de $0{,}5$, a sensibilidade em mulheres cai para **$76\%$** (contra **$89\%$** em homens), gerando uma proporção significativamente maior de **falsos negativos em mulheres e jovens**.
* **Recomendação Clínica:** Para implantação em triagem cardiológica, o limiar de decisão deve ser calibrado especificamente por estrato populacional ou ajustado via curvas de custo clínico, garantindo que pacientes jovens e mulheres com quadros atípicos não sejam subdiagnosticados.
